# Grid, random and Optuna search

This offline notebook demonstrates discrete candidates, continuous ranges, repeated-run objectives, and copying a selected configuration into an experiment. Small budgets teach the API; increase them for a research study.

In [1]:
import numpy as np
import pandas as pd
from pymv.data import MultiViewDataset, simulate_missing_views
from pymv.experiment import Experiment, GridSearchExperiment, RandomSearchExperiment, OptunaSearchExperiment
from pymv.metrics import clustering_accuracy, nmi_score, silhouette_score
from pymv.models import (LMGEC, SCM, IDEC, COMPLETER, DCP, MFLVC, CONAN, DGF,
                         SGLA, UMC, GCoMVKM, MultiViewKMeans, MultiViewSpectralClustering)

def dataset(name, noise, seed):
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(3), 16)
    views = [rng.normal(size=(3, dim))[labels] * 3 +
             rng.normal(scale=noise, size=(48, dim)) for dim in (6, 8)]
    return MultiViewDataset(views, labels=labels, name=name)

clean = dataset("clean", .3, 1)
noisy = dataset("noisy", 1.5, 2)

In [2]:
space = {"n_clusters": [3], "embedding_dim": [3, 4],
         "temperature": [.2, 1., 3.], "max_iter": [3]}
common = dict(model_class=LMGEC, dataset=clean, metrics={"NMI": nmi_score},
              best_metric="NMI", runs=2, seed=42, save_path="experiments/searches")
grid = GridSearchExperiment(param_grid=space, **common).run()
random = RandomSearchExperiment(param_distributions=space, n_iter=3, **common).run()

## Adaptive sampling

Lists mean exact choices. A dictionary with low/high gives a continuous range; `type='int'` gives integers. Optional logarithmic sampling requires positive bounds. Optuna learns from completed mean scores. It does not prune fits.

In [3]:
adaptive = OptunaSearchExperiment(
    param_distributions={
        "n_clusters": [3],
        "embedding_dim": {"type": "int", "low": 2, "high": 4},
        "temperature": {"low": .1, "high": 5., "log": True},
        "max_iter": [3],
    },
    n_iter=4, **common,
)
optuna_result = adaptive.run()
display(pd.DataFrame(optuna_result.report["search"]["steps"]))
optuna_result.export("experiments/searches/optuna.zip")

[I 2026-10-12 14:15:55,594] A new study created in memory with name: no-name-7fbfc1b3-0c15-478a-bb31-af05e2b002dc


,step,parameters,score,best_score_so_far
0,1,"{'n_clusters': 3, 'embedding_dim': 3, 'tempera...",0.73368,0.73368
1,2,"{'n_clusters': 3, 'embedding_dim': 4, 'tempera...",0.73368,0.73368
2,3,"{'n_clusters': 3, 'embedding_dim': 2, 'tempera...",0.73368,0.73368
3,4,"{'n_clusters': 3, 'embedding_dim': 2, 'tempera...",0.73368,0.73368


PosixPath('experiments/searches/optuna.zip')

## Run the selected configuration

Search best parameters are regular constructor values. The search objective is measured on these data, not on an independent held-out test set. Avoid presenting it as unbiased generalization performance.

In [4]:
if optuna_result.best_params:
    best_model = LMGEC(**optuna_result.best_params)
    repeated = Experiment(clean, [best_model], {"ACC": clustering_accuracy, "NMI": nmi_score},
                          runs=3, seed=100, save_path="experiments/best-model").run()

## Search without true labels

Silhouette uses the learned embedding and predicted clusters. Ground-truth labels are ignored when benchmark=False. Davies–Bouldin can also be selected and is minimized automatically.

In [5]:
internal = OptunaSearchExperiment(
    MultiViewKMeans, {"n_clusters": [2, 3, 4], "n_init": [1, 3], "max_iter": [10]},
    clean, metrics={"Silhouette": silhouette_score}, best_metric="Silhouette",
    benchmark=False, n_iter=3, runs=2, seed=42, save_path="experiments/internal-search",
).run()
print(internal.best_params)

[I 2026-10-12 14:15:56,039] A new study created in memory with name: no-name-f75ca079-2c52-4532-9884-5ecb0e5f1844


{'n_clusters': 3, 'n_init': 1, 'max_iter': 10}
